# ارزیابی شبیه‌سازی

پیاده‌سازی محلی منطق `evaluate_deception.py` روی رکوردهای آماده‌شده. شبیه‌ساز به‌جای فراخوانی API از قاعدهٔ همان اسکریپت استفاده می‌کند: گزارش غیرمستقیم یا وجود `miş`/`muş` ← `Deceptive`؛ در غیر این صورت `Truthful`. معیارها نسبت به کلاس مثبت `Deceptive` محاسبه می‌شوند.

In [ ]:
from pathlib import Path
import json
import pandas as pd

DATA_DIR = Path('data')
INPUT = DATA_DIR / 'deception_dataset.csv'
RESULTS_OUT = DATA_DIR / 'simulation_results.csv'
SUMMARY_OUT = DATA_DIR / 'simulation_summary.json'
if not INPUT.exists():
    raise FileNotFoundError(f'{INPUT} not found; run notebook 01 first.')
df = pd.read_csv(INPUT).fillna('')
print('Loaded', len(df), 'records')

## پیاده‌سازی شبیه‌ساز و استانداردسازی برچسب‌ها

In [ ]:
def simulate_llm(record):
    statement = str(record.get('sentence', '')).lower()
    evidentiality = str(record.get('evidentiality', '')).lower()
    if evidentiality == 'indirect_reported' or 'miş' in statement or 'muş' in statement:
        return 'Deceptive'
    return 'Truthful'

def normalize_label(value):
    value = str(value).strip().lower()
    return 'Deceptive' if value in {'deceptive','lie','false'} else 'Truthful'

results = df.copy()
results['raw_response'] = results.apply(lambda r: simulate_llm(r.to_dict()), axis=1)
results['prediction'] = results['raw_response'].map(lambda x: 'Deceptive' if 'deceptive' in x.lower() else 'Truthful')
results['ground_truth'] = results['ground_truth'].map(normalize_label)
results['is_correct'] = results['prediction'].eq(results['ground_truth']).astype(int)
display(results.head())
print('Prediction distribution:', results.prediction.value_counts().to_dict())

## معیارهای کلی و تفکیک‌شده
ماتریس درهم‌ریختگی با قرارداد TP = پیش‌بینی و واقعیت فریبنده محاسبه می‌شود؛ تقسیم بر صفر به صفر نگاشت می‌شود.

In [ ]:
def metrics(frame):
    y = frame['ground_truth']; p = frame['prediction']
    tp = int(((p=='Deceptive') & (y=='Deceptive')).sum())
    fp = int(((p=='Deceptive') & (y!='Deceptive')).sum())
    fn = int(((p!='Deceptive') & (y=='Deceptive')).sum())
    tn = int(((p!='Deceptive') & (y!='Deceptive')).sum())
    precision = tp/(tp+fp) if tp+fp else 0.0
    recall = tp/(tp+fn) if tp+fn else 0.0
    f1 = 2*precision*recall/(precision+recall) if precision+recall else 0.0
    return {'count':len(frame),'correct':int((p==y).sum()),'accuracy':(p==y).mean() if len(frame) else 0.0,
            'TP':tp,'FP':fp,'FN':fn,'TN':tn,'precision':precision,'recall':recall,'f1_deceptive':f1}

overall = metrics(results)
print('Overall:', overall)
breakdowns=[]
for col in ['label_type','source_column']:
    for key, group in results.groupby(col, dropna=False):
        breakdowns.append({'breakdown':col,'condition':key,**metrics(group)})
breakdown_df = pd.DataFrame(breakdowns)
display(breakdown_df)

In [ ]:
RESULTS_OUT.parent.mkdir(parents=True, exist_ok=True)
results.to_csv(RESULTS_OUT,index=False,encoding='utf-8-sig')
with open(SUMMARY_OUT,'w',encoding='utf-8') as f:
    json.dump({'overall':overall,'breakdowns':breakdowns},f,indent=2,ensure_ascii=False)
assert len(pd.read_csv(RESULTS_OUT)) == len(results)
print('Saved:', RESULTS_OUT, 'and', SUMMARY_OUT)